# Register an Azure ML Environment

Generate a Conda file in the notebook, define a versioned environment from it and a base image, register it, and retrieve it for verification.

**Source:** Adapted from [Azure/azureml-examples environment.ipynb](https://github.com/Azure/azureml-examples/blob/7dbe9a3ddfc4a920a9de82eaa4af7eaf118841d8/sdk/python/assets/environment/environment.ipynb), MIT License.

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential, ManagedIdentityCredential
from dotenv import dotenv_values
import textwrap
import yaml
from azure.ai.ml.entities import Environment

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
ENV_KEYS = {
    "AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP",
    "AZUREML_WORKSPACE_NAME", "AZUREML_REGISTRY_NAME",
    "AZUREML_COMPUTE_NAME", "AZUREML_COMPUTE_IDENTITY_CLIENT_ID",
    "USE_MANAGED_IDENTITY",
}
SUBSCRIPTION_ID = (env.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (env.get("AZURE_RESOURCE_GROUP") or "").strip()
WORKSPACE_NAME = (env.get("AZUREML_WORKSPACE_NAME") or "").strip()
REGISTRY_NAME = (env.get("AZUREML_REGISTRY_NAME") or "").strip()
COMPUTE_NAME = (env.get("AZUREML_COMPUTE_NAME") or "").strip() or None
COMPUTE_IDENTITY_CLIENT_ID = (env.get("AZUREML_COMPUTE_IDENTITY_CLIENT_ID") or "").strip() or None

if set(env) - ENV_KEYS:
    raise ValueError(f"Unexpected .env keys: {', '.join(sorted(set(env) - ENV_KEYS))}.")
if not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in the four required workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME, COMPUTE_NAME or "", COMPUTE_IDENTITY_CLIENT_ID or "")):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

use_managed_identity = (env.get("USE_MANAGED_IDENTITY", "true") or "").strip().lower()
if use_managed_identity not in {"true", "false"}:
    raise ValueError("USE_MANAGED_IDENTITY must be true or false in .env.")
credential = ManagedIdentityCredential(process_timeout=60) if use_managed_identity == "true" else AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}



ENVIRONMENT_NAME = ASSET_NAMES["environment"]


In [ ]:
import requests

BASE_IMAGE = "mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
manifest_url = "https://mcr.microsoft.com/v2/azureml/openmpi4.1.0-ubuntu20.04/manifests/latest"
manifest_response = requests.get(
    manifest_url,
    headers={"Accept": "application/vnd.docker.distribution.manifest.v2+json"},
    timeout=30,
 )
if manifest_response.status_code != 200:
    raise RuntimeError(
        f"Base image manifest is unavailable: {BASE_IMAGE} "
        f"(HTTP {manifest_response.status_code})"
    )
print(f"Verified base image manifest: {BASE_IMAGE}")

generated_dir = SUITE_ROOT / "outputs/generated/foundations/environment"
generated_dir.mkdir(parents=True, exist_ok=True)
conda_file = generated_dir / "conda.yaml"
conda_content = textwrap.dedent("""
name: azureml-workshop-foundations
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - azureml-inference-server-http==1.4.1
      - pandas==2.2.3
""").lstrip()
yaml.safe_load(conda_content)
conda_file.write_text(conda_content, encoding="utf-8")
print(f"Generated Conda file: {conda_file}")
print(conda_content)

environment_definition = Environment(
    name=ENVIRONMENT_NAME,
    image=BASE_IMAGE,
    conda_file=str(conda_file),
    description="Small workshop environment for command and endpoint demonstrations",
    tags={"validation": "workspace-readiness", "purpose": "foundations"},
)

registered_environment = ml_client.environments.create_or_update(environment_definition)
verified_environment = ml_client.environments.get(
    ENVIRONMENT_NAME,
    version=registered_environment.version,
)
assert verified_environment.name == ENVIRONMENT_NAME
assert verified_environment.version == registered_environment.version
assert verified_environment.image == BASE_IMAGE
print(f"Registered environment asset: {verified_environment.name}:{verified_environment.version}")
print("Azure ML requested environment image materialization asynchronously.")
print("Asset retrieval confirms registration, not image-build completion; monitor the build status and logs in Azure ML Studio.")


## Expected Result

The notebook-generated Conda specification defines a named, immutable Azure ML environment. Azure ML assigns its version during registration.

Next: `04_register_model.ipynb`.